In [1]:
%pip -q install "transformers==4.57.1" "datasets==4.3.0" "accelerate==1.10.1" "protobuf>=4.25,<7" "scikit-learn==1.7.2" "tensorboard>=2.18,<3" "fastapi>=0.115,<1" "uvicorn>=0.30,<1"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 82.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires hu

In [2]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, random, hashlib, subprocess

drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/train-KLCN')
DATA_DIR = BASE / 'goemotions'
RUN_ID = 'roberta_large_asl_en_01'
ALLOW_DOWNLOAD_IF_EMPTY = True
MODEL_ID = 'FacebookAI/roberta-large'
LOSS_KIND = 'asl'  # Alternative controlled experiment: weighted_bce, using a new RUN_ID.
LOSS_CONFIG = {'kind': LOSS_KIND, 'gamma_neg': 2.0, 'gamma_pos': 0.0, 'clip': 0.05}
CALIBRATION_TARGET = 0.83  # Buffer on calibration only; test criterion stays >0.80.
EVAL_THRESHOLD = 0.3
WEIGHT_DECAY = 0.02
SEED = 42
MAX_LENGTH = 128
EPOCHS = 6
BATCH_SIZE = 2  # If OOM: 1 and GRAD_ACCUM=32, new RUN_ID.
GRAD_ACCUM = 16
LEARNING_RATE = 1e-5
TARGET = 0.80
MIN_COVERAGE = 0.20
MIN_ACCEPTED = 200
LOG_DIR = BASE / 'log' / RUN_ID
OUT_DIR = BASE / 'output' / RUN_ID
for p in [LOG_DIR, OUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)
if (LOG_DIR / 'test_metrics.json').exists():
    print('Run này đã đánh giá test. Bỏ qua bước 4–8; có thể chạy 9–12 để xuất lại/kiểm tra model. Không dùng test để chọn lại tham số.')
print('Data:', DATA_DIR, '\nLog:', LOG_DIR, '\nOutput:', OUT_DIR)


Mounted at /content/drive
Data: /content/drive/MyDrive/train-KLCN/goemotions 
Log: /content/drive/MyDrive/train-KLCN/log/roberta_large_asl_en_01 
Output: /content/drive/MyDrive/train-KLCN/output/roberta_large_asl_en_01


In [3]:
SOURCES = {'data_io.py': '"""Read official TSV, supported CSV, or Hugging Face save_to_disk directory."""\nimport ast\nimport hashlib\nimport json\nimport re\nfrom pathlib import Path\nimport pandas as pd\n\nLABELS = [\'admiration\', \'amusement\', \'anger\', \'annoyance\', \'approval\', \'caring\',\n          \'confusion\', \'curiosity\', \'desire\', \'disappointment\', \'disapproval\',\n          \'disgust\', \'embarrassment\', \'excitement\', \'fear\', \'gratitude\', \'grief\',\n          \'joy\', \'love\', \'nervousness\', \'optimism\', \'pride\', \'realization\', \'relief\',\n          \'remorse\', \'sadness\', \'surprise\', \'neutral\']\n\n\ndef parse_labels(value):\n    if isinstance(value, str):\n        value = value.strip()\n        if value.startswith(\'[\'):\n            value = ast.literal_eval(value)\n        else:\n            value = [x.strip() for x in value.split(\',\')]\n    if isinstance(value, (int, float)):\n        value = [value]\n    result = []\n    for v in value:\n        if isinstance(v, str) and v in LABELS:\n            result.append(LABELS.index(v))\n        else:\n            f = float(v)\n            if not f.is_integer():\n                raise ValueError(f\'Invalid label: {v}\')\n            result.append(int(f))\n    if not result or any(x < 0 or x >= len(LABELS) for x in result):\n        raise ValueError(f\'Invalid labels: {value}\')\n    return sorted(set(result))\n\n\ndef normalize(text):\n    return re.sub(r\'\\s+\', \' \', text.strip()).casefold()\n\n\ndef load_data(data_dir, allow_download=True):\n    from datasets import Dataset, DatasetDict, load_dataset, load_from_disk\n    root = Path(data_dir)\n    root.mkdir(parents=True, exist_ok=True)\n    if (root / \'dataset_dict.json\').exists():\n        ds = load_from_disk(str(root))\n        source = \'local_hf\'\n    else:\n        paths = {}\n        for split, aliases in {\'train\': [\'train\'], \'validation\': [\'validation\', \'dev\', \'val\'],\n                               \'test\': [\'test\']}.items():\n            candidates = [root / (name + ext) for name in aliases for ext in [\'.tsv\', \'.csv\']]\n            found = [p for p in candidates if p.exists()]\n            if len(found) > 1:\n                raise ValueError(f\'Multiple files for {split}: {found}; keep one or change DATA_DIR.\')\n            if found:\n                paths[split] = found[0]\n        if paths:\n            if len(paths) != 3:\n                raise ValueError(\'Need train.tsv/dev.tsv/test.tsv OR train.csv/validation.csv/test.csv.\')\n            parts = {}\n            for split, path in paths.items():\n                if path.suffix == \'.tsv\':\n                    df = pd.read_csv(path, sep=\'\\t\', header=None, names=[\'text\', \'labels\', \'id\'],\n                                     dtype=str, keep_default_na=False)\n                else:\n                    df = pd.read_csv(path, keep_default_na=False)\n                    if \'text\' not in df:\n                        raise ValueError(\'CSV needs text plus labels or all 28 one-hot columns.\')\n                    if \'labels\' not in df and all(c in df for c in LABELS):\n                        a = df[LABELS].apply(pd.to_numeric).to_numpy()\n                        if not ((a == 0) | (a == 1)).all():\n                            raise ValueError(\'One-hot label columns must contain only 0 or 1.\')\n                        df[\'labels\'] = [[j for j, v in enumerate(row) if v == 1] for row in a]\n                if \'labels\' not in df:\n                    raise ValueError(\'Missing labels column.\')\n                rows = []\n                for i, row in df.iterrows():\n                    text = str(row[\'text\']).strip()\n                    if not text:\n                        raise ValueError(f\'Empty text: {path}, row {i}\')\n                    rows.append({\'text\': text, \'labels\': parse_labels(row[\'labels\']),\n                                 \'id\': str(row.get(\'id\', f\'{split}_{i}\'))})\n                parts[split] = Dataset.from_list(rows)\n            ds = DatasetDict(parts)\n            source = \'local_csv_tsv\'\n        elif any(root.iterdir()):\n            raise ValueError(\'DATA_DIR is not empty but no supported split files found. \'\n                             \'Point DATA_DIR at the exact folder; raw goemotions_1/2/3.csv are annotator data, \'\n                             \'not train/validation/test. Do not randomly split individual annotations.\')\n        elif allow_download:\n            ds = load_dataset(\'google-research-datasets/go_emotions\', \'simplified\')\n            ds.save_to_disk(str(root))\n            source = \'downloaded_official_simplified\'\n        else:\n            raise FileNotFoundError(\'No dataset. Set DATA_DIR or allow first download.\')\n    if not all(k in ds for k in [\'train\', \'validation\', \'test\']):\n        raise ValueError(\'Dataset requires train, validation, test splits.\')\n    for split in [\'train\', \'validation\', \'test\']:\n        feature = ds[split].features.get(\'labels\')\n        names = getattr(getattr(feature, \'feature\', None), \'names\', None)\n        if names is not None and list(names) != LABELS:\n            raise ValueError(\'Label order differs from canonical GoEmotions order.\')\n        for row in ds[split]:\n            if not isinstance(row[\'text\'], str) or not row[\'text\'].strip():\n                raise ValueError(f\'Invalid text in {split}\')\n            parse_labels(row[\'labels\'])\n    # Remove exact-text leakage using text only, preserving all test rows.\n    before = {k: len(ds[k]) for k in ds}\n    test_texts = {normalize(t) for t in ds[\'test\'][\'text\']}\n    val_texts = {normalize(t) for t in ds[\'validation\'][\'text\']}\n    ds[\'train\'] = ds[\'train\'].filter(lambda r: normalize(r[\'text\']) not in (test_texts | val_texts))\n    ds[\'validation\'] = ds[\'validation\'].filter(lambda r: normalize(r[\'text\']) not in test_texts)\n    for split in [\'train\', \'validation\', \'test\']:\n        if len(ds[split]) == 0:\n            raise ValueError(f\'Empty split after leakage removal: {split}\')\n    hashes = {}\n    for split in ds:\n        h = hashlib.sha256()\n        for row in ds[split]:\n            h.update(json.dumps([row[\'text\'], parse_labels(row[\'labels\'])], ensure_ascii=False).encode())\n            h.update(b\'\\n\')\n        hashes[split] = h.hexdigest()\n    return ds, {\'source\': source, \'before\': before, \'after\': {k: len(ds[k]) for k in ds},\n                \'sha256\': hashes, \'protocol\': \'official/local splits, exact-text decontamination; test unchanged\'}\n', 'evaluation.py': '"""Metrics for 28-label emotion detection; no fabricated accuracy."""\nimport numpy as np\nfrom sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score\n\n\ndef multilabel_metrics(y, p, thresholds=0.5):\n    pred = p >= np.asarray(thresholds)\n    return {\n        \'micro_f1\': float(f1_score(y, pred, average=\'micro\', zero_division=0)),\n        \'macro_f1\': float(f1_score(y, pred, average=\'macro\', zero_division=0)),\n        \'micro_precision\': float(precision_score(y, pred, average=\'micro\', zero_division=0)),\n        \'micro_recall\': float(recall_score(y, pred, average=\'micro\', zero_division=0)),\n        \'exact_match_accuracy\': float(accuracy_score(y, pred)),\n        \'top1_hit_rate\': float(y[np.arange(len(y)), p.argmax(axis=1)].mean()),\n        \'empty_prediction_rate\': float((pred.sum(axis=1) == 0).mean()),\n    }\n\n\ndef tune_thresholds(y, p):\n    grid = np.round(np.arange(0.05, 0.951, 0.025), 3)\n    scores = [f1_score(y, p >= t, average=\'micro\', zero_division=0) for t in grid]\n    global_t = float(grid[int(np.argmax(scores))])\n    thresholds = np.full(y.shape[1], global_t)\n    # Rare labels retain the global threshold; shrink other estimates toward it.\n    for j in range(y.shape[1]):\n        n = int(y[:, j].sum())\n        if n >= 5:\n            best = float(grid[np.argmax([\n                f1_score(y[:, j], p[:, j] >= t, zero_division=0) for t in grid\n            ])])\n            weight = n / (n + 30.0)\n            thresholds[j] = weight * best + (1 - weight) * global_t\n    return global_t, thresholds.tolist()\n\n\ndef wilson_lower(k, n, z=1.96):\n    if n == 0:\n        return 0.0\n    p = k / n\n    return float((p + z*z/(2*n) - z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))) / (1+z*z/n))\n\n\ndef selective_metrics(y, p, threshold):\n    top = p.argmax(axis=1)\n    accept = p.max(axis=1) >= threshold\n    n = int(accept.sum())\n    k = int(y[np.arange(len(y)), top][accept].sum())\n    return {\'threshold\': float(threshold), \'accepted\': n, \'total\': len(y),\n            \'coverage\': float(n / len(y)), \'top1_precision\': float(k/n) if n else None,\n            \'wilson_95_lower\': wilson_lower(k, n)}\n\n\ndef select_policy(y, p, target=0.8, min_coverage=0.2, min_count=200):\n    rows = [selective_metrics(y, p, t) for t in np.linspace(0.05, 0.99, 189)]\n    eligible = [r for r in rows if r[\'accepted\'] >= min_count\n                and r[\'coverage\'] >= min_coverage and r[\'wilson_95_lower\'] >= target]\n    if eligible:\n        best = max(eligible, key=lambda r: (r[\'coverage\'], -r[\'threshold\']))\n        return {**best, \'enabled\': True, \'target\': target,\n                \'min_coverage\': min_coverage, \'min_count\': min_count}, rows\n    return {\'enabled\': False, \'threshold\': 1.01, \'target\': target,\n            \'min_coverage\': min_coverage, \'min_count\': min_count,\n            \'reason\': \'No calibration threshold meets precision and coverage constraints.\'}, rows\n\n\ndef goal_checks(full, selective, target=0.8):\n    values = {\'micro_f1\': full[\'micro_f1\'], \'macro_f1\': full[\'macro_f1\'],\n              \'top1_all\': full[\'top1_hit_rate\'],\n              \'top1_accepted\': selective[\'top1_precision\'],\n              \'accepted_wilson_95_lower\': selective[\'wilson_95_lower\']}\n    flags = {k: bool(v is not None and v > target) for k, v in values.items()}\n    return values, flags\n', 'losses.py': '"""Experimental positive-weighted asymmetric loss. No claim of GoEmotions accuracy."""\nimport torch\nimport torch.nn.functional as F\nfrom transformers import Trainer\n\n\ndef multilabel_loss(logits, labels, pos_weight, kind=\'asl\', gamma_neg=2.0,\n                    gamma_pos=0.0, clip=0.05):\n    # Compute logarithms in FP32 under mixed precision.\n    logits = logits.float()\n    labels = labels.float()\n    weights = pos_weight.to(device=logits.device, dtype=torch.float32)\n    if kind == \'weighted_bce\':\n        return F.binary_cross_entropy_with_logits(logits, labels, pos_weight=weights)\n    if kind != \'asl\':\n        raise ValueError(\'kind must be asl or weighted_bce\')\n    positive = torch.sigmoid(logits)\n    negative = (1.0 - positive + clip).clamp(max=1.0)\n    log_pos = F.logsigmoid(logits)\n    log_neg = negative.clamp(min=1e-8).log()\n    # Detach focusing weights, following the common ASL implementation variant.\n    with torch.no_grad():\n        focus_pos = (1.0 - positive).pow(gamma_pos)\n        focus_neg = (1.0 - negative).pow(gamma_neg)\n    return -(labels * weights * focus_pos * log_pos +\n             (1.0 - labels) * focus_neg * log_neg).mean()\n\n\nclass EmotionTrainer(Trainer):\n    def __init__(self, *args, pos_weight, loss_config, **kwargs):\n        super().__init__(*args, **kwargs)\n        self.pos_weight = torch.as_tensor(pos_weight, dtype=torch.float32)\n        self.loss_config = dict(loss_config)\n        # We return a per-microbatch mean and do not consume num_items_in_batch.\n        # Trainer must retain its gradient-accumulation normalization.\n        self.model_accepts_loss_kwargs = False\n\n    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):\n        labels = inputs[\'labels\']\n        outputs = model(**{k: v for k, v in inputs.items() if k != \'labels\'})\n        loss = multilabel_loss(outputs.logits, labels, self.pos_weight, **self.loss_config)\n        return (loss, outputs) if return_outputs else loss\n', 'predictor.py': '"""English-only exported model; use from a server, not from a public Colab URL."""\nimport json\nfrom pathlib import Path\nimport torch\nfrom transformers import AutoTokenizer, AutoModelForSequenceClassification\n\nclass EmotionPredictor:\n    def __init__(self, model_dir, device=None):\n        root = Path(model_dir)\n        self.config = json.loads((root / \'inference_config.json\').read_text())\n        self.device = device or (\'cuda\' if torch.cuda.is_available() else \'cpu\')\n        self.tokenizer = AutoTokenizer.from_pretrained(root)\n        self.model = AutoModelForSequenceClassification.from_pretrained(root).to(self.device).eval()\n        actual = [self.model.config.id2label[i] for i in range(28)]\n        if actual != self.config[\'labels\']:\n            raise ValueError(\'Unexpected label order\')\n\n    def predict(self, text):\n        if not isinstance(text, str) or not text.strip() or len(text) > 4000:\n            raise ValueError(\'Provide 1–4000 characters of English text.\')\n        text = text.strip()\n        max_length = self.config[\'max_length\']\n        n_tokens = len(self.tokenizer(text, truncation=False)[\'input_ids\'])\n        if n_tokens > max_length:\n            raise ValueError(f\'Input exceeds {max_length} model tokens; shorten the text.\')\n        inputs = self.tokenizer(text, return_tensors=\'pt\').to(self.device)\n        with torch.inference_mode():\n            p = self.model(**inputs).logits.float().sigmoid()[0].cpu().tolist()\n        labels = self.config[\'labels\']\n        order = sorted(range(len(p)), key=lambda i: p[i], reverse=True)\n        top = order[0]\n        policy = self.config[\'selective_policy\']\n        accepted = policy[\'enabled\'] and p[top] >= policy[\'threshold\']\n        return {\n            \'model_version\': self.config[\'run_id\'], \'language\': \'en\',\n            \'status\': \'accepted\' if accepted else \'needs_confirmation\',\n            \'primary_emotion\': labels[top] if accepted else None,\n            \'suggested_emotion\': labels[top], \'score\': float(p[top]),\n            \'score_type\': \'uncalibrated_sigmoid_not_accuracy\',\n            \'requires_confirmation\': not accepted,\n            \'predicted_labels\': [labels[i] for i in order if p[i] >= self.config[\'thresholds\'][i]],\n            \'candidates\': [{\'label\': labels[i], \'score\': float(p[i])} for i in order[:5]],\n            \'emotion_scores\': dict(zip(labels, p)),\n            \'all_five_above_80_in_report\': self.config[\'all_five_above_80\'],\n        }\n', 'api.py': "import os\nfrom contextlib import asynccontextmanager\nfrom threading import Lock\nfrom fastapi import FastAPI, HTTPException\nfrom pydantic import BaseModel, Field\nfrom predictor import EmotionPredictor\n\nlock = Lock()\n@asynccontextmanager\nasync def lifespan(app):\n    app.state.predictor = EmotionPredictor(os.environ.get('MODEL_DIR', '../best_model'))\n    yield\napp = FastAPI(title='English Emotion API', lifespan=lifespan)\nclass Request(BaseModel):\n    text: str = Field(min_length=1, max_length=4000)\n@app.get('/health')\ndef health():\n    return {'status':'ok', 'model_version':app.state.predictor.config['run_id']}\n@app.post('/predict')\ndef predict(req: Request):\n    try:\n        with lock:\n            return app.state.predictor.predict(req.text)\n    except ValueError as e:\n        raise HTTPException(status_code=422, detail=str(e)) from e\n", 'requirements.txt': 'torch>=2.6,<3\ntransformers==4.57.1\nfastapi>=0.115,<1\nuvicorn>=0.30,<1\n'}
RUNTIME = Path('/content/goemotions_english_runtime')
RUNTIME.mkdir(exist_ok=True)
for name, content in SOURCES.items():
    (RUNTIME / name).write_text(content, encoding='utf-8')
sys.path.insert(0, str(RUNTIME))
print('English-only runtime ready.')


English-only runtime ready.


In [4]:
import numpy as np
import pandas as pd
import torch
import transformers, datasets
from packaging.version import Version
from transformers import set_seed
from data_io import LABELS, load_data
from evaluation import multilabel_metrics, tune_thresholds, select_policy, selective_metrics
from scipy.special import expit

assert torch.cuda.is_available(), 'Chọn Runtime → Change runtime type → GPU trước khi train.'
assert Version(torch.__version__.split('+')[0]) >= Version('2.6'), 'Cần Colab runtime có torch >= 2.6.'
set_seed(SEED)
print(torch.cuda.get_device_name(0), ' | Torch', torch.__version__, '| Transformers', transformers.__version__)
raw, audit = load_data(DATA_DIR, ALLOW_DOWNLOAD_IF_EMPTY)
# Split validation by normalized text group: repeated text cannot cross selection/calibration.
from sklearn.model_selection import GroupShuffleSplit
from data_io import normalize
v = raw['validation']
selection_idx, calibration_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED)
                                      .split(np.arange(len(v)), groups=[normalize(t) for t in v['text']]))
selection = v.select(selection_idx.tolist())
calibration = v.select(calibration_idx.tolist())
assert len(selection) >= 200 and len(calibration) >= 200, 'Validation quá nhỏ cho protocol này.'
print('Train:', len(raw['train']), 'Selection:', len(selection), 'Calibration:', len(calibration), 'Test:', len(raw['test']))
audit['validation_parts'] = {'selection': len(selection), 'calibration': len(calibration)}
(LOG_DIR / 'data_audit.json').write_text(json.dumps(audit, indent=2))
counts = np.zeros(28, dtype=int)
for row in raw['train']:
    counts[row['labels']] += 1
pd.DataFrame({'label': LABELS, 'train_count': counts}).to_csv(LOG_DIR / 'label_counts.csv', index=False)
display(pd.DataFrame({'label': LABELS, 'train_count': counts}))

# Smooth positive weights from TRAIN counts only, capped to limit false positives.
positive_weights = np.clip(np.sqrt(np.median(counts[counts > 0]) / np.maximum(counts, 1)), 1.0, 4.0)
pd.DataFrame({'label': LABELS, 'train_count': counts, 'positive_weight': positive_weights}).to_csv(
    LOG_DIR / 'train_label_weights.csv', index=False)
print('Positive weight range:', positive_weights.min(), positive_weights.max())


Tesla T4  | Torch 2.11.0+cu130 | Transformers 4.57.1


Filter:   0%|          | 0/43410 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5426 [00:00<?, ? examples/s]

Train: 43252 Selection: 2707 Calibration: 2707 Test: 5427


,label,train_count
0,admiration,4120
1,amusement,2325
2,anger,1565
3,annoyance,2467
4,approval,2934
5,caring,1085
6,confusion,1363
7,curiosity,2186
8,desire,641
9,disappointment,1266


Positive weight range: 1.0 4.0


In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from datasets import DatasetDict
from huggingface_hub import model_info
config_path = OUT_DIR / 'run_config.json'
previous = json.loads(config_path.read_text()) if config_path.exists() else None
# Record exact model commit for reproducibility; reuse on resume.
MODEL_REVISION = previous['model_revision'] if previous else model_info(MODEL_ID).sha
run_config = {'run_id': RUN_ID, 'model_id': MODEL_ID, 'model_revision': MODEL_REVISION,
              'seed': SEED, 'max_length': MAX_LENGTH, 'epochs': EPOCHS,
              'batch_size': BATCH_SIZE, 'gradient_accumulation': GRAD_ACCUM,
              'learning_rate': LEARNING_RATE, 'target': TARGET,
              'loss_config': LOSS_CONFIG, 'positive_weights': positive_weights.tolist(),
              'eval_threshold': EVAL_THRESHOLD, 'weight_decay': WEIGHT_DECAY,
              'calibration_target': CALIBRATION_TARGET,
              'min_coverage': MIN_COVERAGE, 'min_accepted': MIN_ACCEPTED,
              'data_sha256': audit['sha256'], 'labels': LABELS,
              'source_sha256': hashlib.sha256(json.dumps(SOURCES, sort_keys=True).encode()).hexdigest()}
if previous and previous != run_config:
    raise RuntimeError('Cấu hình/dữ liệu khác run đã lưu. Đổi RUN_ID để tránh resume sai.')
config_path.write_text(json.dumps(run_config, indent=2))
(LOG_DIR / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION, use_fast=True)

def encode(batch):
    enc = tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)
    target = np.zeros((len(batch['labels']), len(LABELS)), dtype=np.float32)
    for i, indices in enumerate(batch['labels']):
        target[i, indices] = 1.0
    enc['labels'] = target.tolist()
    return enc

parts = DatasetDict({'train': raw['train'], 'selection': selection, 'calibration': calibration})
encoded = DatasetDict({name: ds.map(encode, batched=True, remove_columns=ds.column_names)
                       for name, ds in parts.items()})
padder = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)

def collate(features):
    labels = torch.tensor([f['labels'] for f in features], dtype=torch.float32)
    batch = padder([{k: v for k, v in f.items() if k != 'labels'} for f in features])
    batch['labels'] = labels
    return batch

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, num_labels=28,
    id2label=dict(enumerate(LABELS)), label2id={v: k for k, v in enumerate(LABELS)},
    problem_type='multi_label_classification')
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/43252 [00:00<?, ? examples/s]

Map:   0%|          | 0/2707 [00:00<?, ? examples/s]

Map:   0%|          | 0/2707 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
if (LOG_DIR / 'test_metrics.json').exists():
    raise RuntimeError('Run đã đánh giá test. Bỏ qua bước 6–8, dùng model đã xuất hoặc RUN_ID mới.')
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback, TrainerCallback
from transformers.trainer_utils import get_last_checkpoint
from losses import EmotionTrainer, multilabel_loss

# Numerical/backward smoke test; no training-data change.
x = torch.tensor([[-30., 0., 30.], [2., -2., 0.]], requires_grad=True)
y = torch.tensor([[0., 1., 1.], [1., 0., 0.]])
w = torch.tensor([1., 2., 4.])
for kind in ['asl', 'weighted_bce']:
    z = x.detach().clone().requires_grad_(True)
    loss = multilabel_loss(z, y, w, kind=kind)
    loss.backward()
    assert torch.isfinite(loss) and torch.isfinite(z.grad).all()
# ASL with no focusing or clipping must equal weighted BCE.
a = multilabel_loss(x, y, w, kind='asl', gamma_neg=0., gamma_pos=0., clip=0.)
b = multilabel_loss(x, y, w, kind='weighted_bce')
assert torch.allclose(a, b, atol=1e-5), (a, b)
print('Loss numerical checks passed.')

class JsonLogger(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        row = {'step': state.global_step, 'epoch': state.epoch,
               'utc': datetime.now(timezone.utc).isoformat(), **(logs or {})}
        with (LOG_DIR / 'training.jsonl').open('a') as f:
            f.write(json.dumps(row, default=float) + '\n')

def compute_metrics(result):
    p = expit(result.predictions)
    metrics = multilabel_metrics(result.label_ids, p, EVAL_THRESHOLD)
    fixed = multilabel_metrics(result.label_ids, p, 0.5)
    metrics['micro_f1_at_05'] = fixed['micro_f1']
    metrics['macro_f1_at_05'] = fixed['macro_f1']
    metrics['balanced_goal_score'] = min(metrics['micro_f1'], metrics['macro_f1'], metrics['top1_hit_rate'])
    return metrics

use_bf16 = torch.cuda.is_bf16_supported()
args = TrainingArguments(
    output_dir=str(OUT_DIR / 'checkpoints'), logging_dir=str(LOG_DIR / 'tensorboard'),
    num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM, eval_accumulation_steps=16,
    weight_decay=WEIGHT_DECAY, warmup_ratio=0.1, lr_scheduler_type='cosine',
    max_grad_norm=1.0, optim='adamw_torch',
    eval_strategy='epoch', save_strategy='epoch', logging_steps=25,
    load_best_model_at_end=True, metric_for_best_model='balanced_goal_score', greater_is_better=True,
    save_total_limit=2, save_safetensors=True,
    bf16=use_bf16, fp16=not use_bf16, gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    report_to=['tensorboard'], seed=SEED, data_seed=SEED, dataloader_num_workers=2)
trainer = EmotionTrainer(pos_weight=positive_weights, loss_config=LOSS_CONFIG, model=model, args=args, train_dataset=encoded['train'],
                  eval_dataset=encoded['selection'], processing_class=tokenizer,
                  data_collator=collate, compute_metrics=compute_metrics,
                  callbacks=[EarlyStoppingCallback(early_stopping_patience=2), JsonLogger()])
checkpoint_dir = OUT_DIR / 'checkpoints'
last = get_last_checkpoint(str(checkpoint_dir)) if checkpoint_dir.exists() else None
print('Resume:', last or 'new run')
train_result = trainer.train(resume_from_checkpoint=last)
trainer.save_state()
BEST_DIR = OUT_DIR / 'best_model'
trainer.save_model(str(BEST_DIR))
tokenizer.save_pretrained(str(BEST_DIR))
(LOG_DIR / 'training_summary.json').write_text(json.dumps(train_result.metrics, indent=2))
print('Best checkpoint:', trainer.state.best_model_checkpoint)

pd.DataFrame(trainer.state.log_history).to_csv(LOG_DIR / 'training_history.csv', index=False)


Loss numerical checks passed.
Resume: new run


Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Micro Precision,Micro Recall,Exact Match Accuracy,Top1 Hit Rate,Empty Prediction Rate,Micro F1 At 05,Macro F1 At 05,Balanced Goal Score
1,0.046400,0.043854,0.348219,0.311411,0.214580,0.923149,0.031031,0.615072,0.000000,0.562687,0.497773,0.311411


In [ ]:
if (LOG_DIR / 'test_metrics.json').exists():
    raise RuntimeError('Không chọn lại ngưỡng sau khi test. Dùng calibration_metrics.json đã lưu.')
cal_pred = trainer.predict(encoded['calibration'])
y_cal = cal_pred.label_ids.astype(int)
p_cal = expit(cal_pred.predictions)
global_t, per_label = tune_thresholds(y_cal, p_cal)
choices = {'fixed_0.5': [0.5]*28, 'global': [global_t]*28, 'per_label_shrunk': per_label}
cal_metrics = {name: multilabel_metrics(y_cal, p_cal, ts) for name, ts in choices.items()}
threshold_method = max(choices, key=lambda k: min(cal_metrics[k]['micro_f1'], cal_metrics[k]['macro_f1']))
thresholds = choices[threshold_method]
policy, sweep = select_policy(y_cal, p_cal, CALIBRATION_TARGET, MIN_COVERAGE, MIN_ACCEPTED)
pd.DataFrame(sweep).to_csv(LOG_DIR / 'calibration_precision_coverage.csv', index=False)
pd.DataFrame({'label': LABELS, 'threshold': thresholds}).to_csv(LOG_DIR / 'thresholds.csv', index=False)
(LOG_DIR / 'calibration_metrics.json').write_text(json.dumps(
    {'methods': cal_metrics, 'selected': threshold_method, 'selective_policy': policy}, indent=2))
np.savez_compressed(LOG_DIR / 'calibration_predictions.npz', labels=y_cal, probabilities=p_cal)
print('Threshold method:', threshold_method)
print('Calibration policy:', json.dumps(policy, indent=2))

print('Calibration multi-label metrics:')
print(json.dumps(cal_metrics, indent=2))


In [ ]:
from sklearn.metrics import classification_report
if (LOG_DIR / 'test_metrics.json').exists():
    raise RuntimeError('Đã đánh giá test cho run này; dùng báo cáo đã lưu.')
test_encoded = raw['test'].map(encode, batched=True, remove_columns=raw['test'].column_names)
test_pred = trainer.predict(test_encoded)
y_test = test_pred.label_ids.astype(int)
p_test = expit(test_pred.predictions)
full_metrics = multilabel_metrics(y_test, p_test, thresholds)
selective = selective_metrics(y_test, p_test, policy['threshold'])
selective_met = bool(policy['enabled'] and selective['accepted'] >= MIN_ACCEPTED
                     and selective['coverage'] >= MIN_COVERAGE
                     and selective['wilson_95_lower'] > TARGET)
report = {'run_id': RUN_ID, 'protocol': audit['protocol'],
          'full_test': full_metrics,
          'fixed_0_5_reference': multilabel_metrics(y_test, p_test, 0.5),
          'selective_top1': selective,
          'primary_goal_met': bool(full_metrics['micro_f1'] >= TARGET),
          'selective_goal_met': selective_met,
          'language_evaluated': 'en', 'test_status': 'previously_seen_benchmark',
          'va_regression_evaluated': False}
np.savez_compressed(LOG_DIR / 'test_predictions.npz', labels=y_test, probabilities=p_test)
pd.DataFrame(classification_report(y_test, p_test >= np.asarray(thresholds),
    target_names=LABELS, output_dict=True, zero_division=0)).T.to_csv(LOG_DIR / 'per_label_report.csv')
wrong = np.flatnonzero(y_test[np.arange(len(y_test)), p_test.argmax(1)] == 0)[:50]
pd.DataFrame([{'text': raw['test'][int(i)]['text'],
               'true_labels': ','.join(LABELS[j] for j in np.flatnonzero(y_test[i])),
               'predicted_top1': LABELS[int(p_test[i].argmax())]} for i in wrong]
            ).to_csv(LOG_DIR / 'error_examples.csv', index=False)
from evaluation import goal_checks
values, flags = goal_checks(full_metrics, selective, TARGET)
report['five_metric_values'] = values
report['five_metric_pass_flags'] = flags
report['all_five_above_80'] = bool(all(flags.values()) and selective_met)
report['coverage_constraint_met'] = bool(selective['coverage'] >= MIN_COVERAGE and selective['accepted'] >= MIN_ACCEPTED)
# Write report after all evaluation exports complete.
(LOG_DIR / 'test_metrics.json').write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
print('Mục tiêu micro-F1 80%:', 'ĐẠT' if report['primary_goal_met'] else 'CHƯA ĐẠT')
print('Mục tiêu top-1 chọn lọc:', 'ĐẠT' if selective_met else 'CHƯA ĐẠT — yêu cầu xác nhận trong app')

print('TẤT CẢ 5 CHỈ SỐ >80%:', 'ĐẠT' if report['all_five_above_80'] else 'CHƯA ĐẠT')


In [ ]:
import shutil
import pandas as pd
from data_io import LABELS
BEST_DIR = OUT_DIR / 'best_model'
report = json.loads((LOG_DIR / 'test_metrics.json').read_text())
cal_saved = json.loads((LOG_DIR / 'calibration_metrics.json').read_text())
run_saved = json.loads((OUT_DIR / 'run_config.json').read_text())
threshold_table = pd.read_csv(LOG_DIR / 'thresholds.csv')
assert threshold_table['label'].tolist() == LABELS
policy = dict(cal_saved['selective_policy'])
policy['enabled'] = report['selective_goal_met']
if not policy['enabled']:
    policy['reason'] = 'Calibration or test selective requirement not met.'
config = {'run_id':RUN_ID, 'labels':LABELS, 'thresholds':threshold_table['threshold'].tolist(),
          'max_length':run_saved['max_length'], 'selective_policy':policy,
          'language':'en', 'all_five_above_80':report['all_five_above_80'],
          'test_status':report['test_status']}
(BEST_DIR / 'inference_config.json').write_text(json.dumps(config, indent=2))
(BEST_DIR / 'evaluation_summary.json').write_text(json.dumps(report, indent=2))
shutil.copy2(OUT_DIR / 'run_config.json', BEST_DIR / 'run_config.json')
DEPLOY_DIR = OUT_DIR / 'deployment'
DEPLOY_DIR.mkdir(exist_ok=True)
for name in ['predictor.py', 'api.py', 'requirements.txt']:
    (DEPLOY_DIR / name).write_text(SOURCES[name], encoding='utf-8')
print('Exported model:', BEST_DIR)
print('Exported API:', DEPLOY_DIR)


In [ ]:
import matplotlib.pyplot as plt
history = (pd.DataFrame(trainer.state.log_history) if 'trainer' in globals() else
           pd.read_json(LOG_DIR / 'training.jsonl', lines=True))
history.to_csv(LOG_DIR / 'training_history.csv', index=False)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, key, title in zip(axes, ['loss', 'eval_loss', 'eval_micro_f1'],
                          ['Training loss', 'Selection loss', 'Selection micro-F1 @ 0.3']):
    if key in history:
        rows = history.dropna(subset=[key])
        ax.plot(rows['epoch'], rows[key], marker='o', markersize=3)
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.grid(alpha=0.25)
fig.tight_layout(); fig.savefig(LOG_DIR / 'training_curves.png', dpi=160); plt.show()
curve = pd.read_csv(LOG_DIR / 'calibration_precision_coverage.csv')
fig, ax = plt.subplots(figsize=(6,4))
ax.plot(curve['coverage'], curve['top1_precision'], label='Calibration precision')
ax.axhline(TARGET, color='red', linestyle='--', label='Target 0.80')
ax.set(xlabel='Coverage', ylabel='Top-1 precision', ylim=(0,1.02), xlim=(0,1))
ax.legend(); ax.grid(alpha=0.25); fig.tight_layout()
fig.savefig(LOG_DIR / 'calibration_precision_coverage.png', dpi=160); plt.show()


In [ ]:
import gc
import torch
if 'trainer' in globals():
    trainer.model.to('cpu')
    del trainer
if 'model' in globals():
    del model
gc.collect(); torch.cuda.empty_cache()
from predictor import EmotionPredictor
predictor = EmotionPredictor(OUT_DIR / 'best_model')
for text in ['I feel happy and grateful today.', 'Today, I do not feel very happy.',
             'I miss my friend, but I am excited about our next meeting.']:
    print(json.dumps(predictor.predict(text), indent=2))


In [ ]:
# Optional; reuses existing files, does not retrain.
zip_path = shutil.make_archive(str(OUT_DIR / 'english_emotion_deploy'), 'zip', root_dir=OUT_DIR,
                               base_dir='best_model')
import zipfile
with zipfile.ZipFile(zip_path, 'a', zipfile.ZIP_DEFLATED) as z:
    for path in DEPLOY_DIR.iterdir():
        if path.is_file(): z.write(path, f'deployment/{path.name}')
print('Deployment ZIP:', zip_path)
print('Báo cáo cần gửi lại để kiểm tra chất lượng:', LOG_DIR / 'test_metrics.json',
      LOG_DIR / 'per_label_report.csv', LOG_DIR / 'training_curves.png', sep='\n')
